In [ ]:
# =======================
# Sleeper DLT Ingestion — BRONZE (with league_info enabled)
# Snapshots: rosters, users, transactions, matchups, drafts, draft_picks, draft_order, league_info
# =======================

import dlt
import time
import json
import requests
from typing import Any, List, Optional
from pyspark.sql import functions as F, types as T

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------
REQUEST_TIMEOUT_SECONDS = 20
RETRY_ATTEMPTS = 3
RETRY_BACKOFF_SECONDS = 2.0
# Gentle rate limiting to avoid 429s; tune if needed.
INTER_REQUEST_SLEEP_SECONDS = 0.12  # ~8–9 req/sec

# Your table that lists leagues to ingest:
LEAGUE_TABLE = "workspace.sleeper_raw.leagues"  # must have column: league_id STRING

# NFL regular season weeks (adjust for preseason/postseason or multi-season logic)
NFL_WEEKS = list(range(1, 19))

# Helper to build an empty DF in DLT-safe way
def empty_df(schema):
    # No sparkContext, no RDD — fully whitelisted
    return spark.createDataFrame([], schema)

# ------------------------------------------------------------
# HTTP helpers
# ------------------------------------------------------------
def http_get_json(url: str) -> Optional[Any]:
    """GET with retries/backoff; returns parsed JSON or None."""
    sess = requests.Session()
    last_exc = None
    for attempt in range(1, RETRY_ATTEMPTS + 1):
        try:
            resp = sess.get(url, timeout=REQUEST_TIMEOUT_SECONDS)
            if resp.status_code == 429 or 500 <= resp.status_code < 600:
                raise requests.HTTPError(f"{resp.status_code} {resp.text}")
            resp.raise_for_status()
            return resp.json()
        except Exception as e:
            last_exc = e
            time.sleep(RETRY_BACKOFF_SECONDS * attempt)
    print(f"[WARN] GET failed after retries: {url} :: {last_exc}")
    return None

# ------------------------------------------------------------
# Normalizers
# ------------------------------------------------------------
def as_list_str(v):
    if not v:
        return []
    if isinstance(v, list):
        return ["" if x is None else str(x) for x in v]
    return [str(v)]

def as_str(v):
    return None if v is None else str(v)

def as_bool_or_none(v):
    if v is None:
        return None
    if isinstance(v, bool):
        return v
    s = str(v).strip().lower()
    if s in ("true","1","yes","y"):
        return True
    if s in ("false","0","no","n"):
        return False
    return None

def as_str_map(v):
    if not isinstance(v, dict):
        return {}
    out = {}
    for k, val in v.items():
        out[str(k)] = "" if val is None else str(val)
    return out

def as_int(v):
    if v in (None, ""):
        return None
    try:
        return int(float(v))
    except Exception:
        return None

def as_long(v):
    if v in (None, ""):
        return None
    try:
        return int(v)
    except Exception:
        try:
            return int(float(v))
        except Exception:
            return None

def get_league_ids() -> List[str]:
    return [r["league_id"] for r in spark.read.table(LEAGUE_TABLE).select("league_id").collect()]

# ------------------------------------------------------------
# Schemas (explicit to avoid inference issues)
# ------------------------------------------------------------
ROSTERS_SCHEMA = T.StructType([
    T.StructField("league_id", T.StringType(), True),
    T.StructField("roster_id", T.IntegerType(), True),
    T.StructField("owner_id", T.StringType(), True),
    T.StructField("co_owners", T.ArrayType(T.StringType()), True),
    T.StructField("keepers", T.ArrayType(T.StringType()), True),
    T.StructField("players", T.ArrayType(T.StringType()), True),
    T.StructField("reserve", T.ArrayType(T.StringType()), True),
    T.StructField("starters", T.ArrayType(T.StringType()), True),
    T.StructField("taxi", T.ArrayType(T.StringType()), True),
    T.StructField("player_map", T.MapType(T.StringType(), T.StringType()), True),
    T.StructField("metadata", T.MapType(T.StringType(), T.StringType()), True),
    T.StructField("settings", T.StructType([
        T.StructField("division", T.IntegerType(), True),
        T.StructField("fpts", T.IntegerType(), True),
        T.StructField("fpts_against", T.IntegerType(), True),
        T.StructField("fpts_against_decimal", T.IntegerType(), True),
        T.StructField("fpts_decimal", T.IntegerType(), True),
        T.StructField("losses", T.IntegerType(), True),
        T.StructField("ppts", T.IntegerType(), True),
        T.StructField("ppts_decimal", T.IntegerType(), True),
        T.StructField("ties", T.IntegerType(), True),
        T.StructField("total_moves", T.IntegerType(), True),
        T.StructField("waiver_budget_used", T.IntegerType(), True),
        T.StructField("waiver_position", T.IntegerType(), True),
        T.StructField("wins", T.IntegerType(), True),
    ]), True),
])

USERS_SCHEMA = T.StructType([
    T.StructField("league_id", T.StringType(), True),
    T.StructField("user_id", T.StringType(), True),
    T.StructField("display_name", T.StringType(), True),
    T.StructField("metadata", T.MapType(T.StringType(), T.StringType()), True),
    T.StructField("is_owner", T.BooleanType(), True),
])

TRANSACTIONS_SCHEMA = T.StructType([
    T.StructField("league_id", T.StringType(), True),
    T.StructField("transaction_id", T.StringType(), True),
    T.StructField("type", T.StringType(), True),           # trade, waiver, free_agent, etc.
    T.StructField("status", T.StringType(), True),
    T.StructField("creator", T.StringType(), True),
    T.StructField("adds", T.MapType(T.StringType(), T.IntegerType()), True),    # player_id -> roster_id
    T.StructField("drops", T.MapType(T.StringType(), T.IntegerType()), True),   # player_id -> roster_id
    T.StructField("roster_ids", T.ArrayType(T.IntegerType()), True),
    T.StructField("draft_picks", T.ArrayType(T.StringType()), True),            # JSON strings for flexibility
    T.StructField("waiver_bid", T.IntegerType(), True),
    T.StructField("leg", T.IntegerType(), True),          # week
    T.StructField("metadata", T.MapType(T.StringType(), T.StringType()), True),
    T.StructField("created", T.LongType(), True),         # epoch ms
])

MATCHUPS_SCHEMA = T.StructType([
    T.StructField("league_id", T.StringType(), True),
    T.StructField("week", T.IntegerType(), True),
    T.StructField("matchup_id", T.IntegerType(), True),
    T.StructField("roster_id", T.IntegerType(), True),
    T.StructField("points", T.DoubleType(), True),
    T.StructField("starters", T.ArrayType(T.StringType()), True),
    T.StructField("players", T.ArrayType(T.StringType()), True),
    T.StructField("players_points", T.MapType(T.StringType(), T.DoubleType()), True),
])

DRAFTS_SCHEMA = T.StructType([
    T.StructField("league_id", T.StringType(), True),
    T.StructField("draft_id", T.StringType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("type", T.StringType(), True),
    T.StructField("season", T.StringType(), True),
    T.StructField("metadata", T.MapType(T.StringType(), T.StringType()), True),
    T.StructField("settings", T.MapType(T.StringType(), T.StringType()), True),
])

DRAFT_PICKS_SCHEMA = T.StructType([
    T.StructField("draft_id", T.StringType(), True),
    T.StructField("league_id", T.StringType(), True),
    T.StructField("pick_no", T.IntegerType(), True),
    T.StructField("round", T.IntegerType(), True),
    T.StructField("roster_id", T.IntegerType(), True),
    T.StructField("player_id", T.StringType(), True),
    T.StructField("picked_by", T.StringType(), True),  # user_id
    T.StructField("is_keeper", T.BooleanType(), True),
    T.StructField("metadata", T.MapType(T.StringType(), T.StringType()), True),
])

DRAFT_SLOT_TO_ROSTER_SCHEMA = T.StructType([
    T.StructField("draft_id", T.StringType(), True),
    T.StructField("league_id", T.StringType(), True),
    T.StructField("slot", T.IntegerType(), True),
    T.StructField("roster_id", T.IntegerType(), True),
])

LEAGUE_INFO_SCHEMA = T.StructType([
    T.StructField("league_id", T.StringType(), True),
    T.StructField("name", T.StringType(), True),
    T.StructField("season", T.StringType(), True),
    T.StructField("sport", T.StringType(), True),
    T.StructField("avatar", T.StringType(), True),
    T.StructField("settings", T.MapType(T.StringType(), T.StringType()), True),
    T.StructField("metadata", T.MapType(T.StringType(), T.StringType()), True),
    T.StructField("roster_positions", T.ArrayType(T.StringType()), True),
])

# ------------------------------------------------------------
# BRONZE SNAPSHOTS
# ------------------------------------------------------------

@dlt.table(
    name="sleeper_league_info_snapshot",
    comment="League metadata/settings snapshot per run"
)
def sleeper_league_info_snapshot():
    rows = []
    for lg in get_league_ids():
        url = f"https://api.sleeper.app/v1/league/{lg}"
        data = http_get_json(url) or {}
        time.sleep(INTER_REQUEST_SLEEP_SECONDS)
        if data:
            rows.append({
                "league_id": as_str(lg),
                "name": as_str(data.get("name")),
                "season": as_str(data.get("season")),
                "sport": as_str(data.get("sport")),
                "avatar": as_str(data.get("avatar")),
                "settings": as_str_map(data.get("settings")),
                "metadata": as_str_map(data.get("metadata")),
                "roster_positions": data.get("roster_positions") or [],
            })
    if not rows:
        return empty_df(LEAGUE_INFO_SCHEMA)
    return spark.createDataFrame(rows, schema=LEAGUE_INFO_SCHEMA) \
                 .withColumn("_ingested_at", F.current_timestamp())

@dlt.table(
    name="sleeper_rosters_snapshot",
    comment="Roster snapshot per league (point-in-time)"
)
@dlt.expect_or_drop("league_id_present", "league_id IS NOT NULL")
def sleeper_rosters_snapshot():
    rows = []
    for lg in get_league_ids():
        url = f"https://api.sleeper.app/v1/league/{lg}/rosters"
        data = http_get_json(url) or []
        time.sleep(INTER_REQUEST_SLEEP_SECONDS)
        for r in data:
            settings = r.get("settings") or {}
            rows.append({
                "league_id": as_str(lg),
                "roster_id": as_int(r.get("roster_id")),
                "owner_id": as_str(r.get("owner_id")),
                "co_owners": as_list_str(r.get("co_owners")),
                "keepers": as_list_str(r.get("keepers")),
                "players": as_list_str(r.get("players")),
                "reserve": as_list_str(r.get("reserve")),
                "starters": as_list_str(r.get("starters")),
                "taxi": as_list_str(r.get("taxi")),
                "player_map": as_str_map(r.get("player_map")),
                "metadata": as_str_map(r.get("metadata")),
                "settings": {
                    "division": as_int(settings.get("division")),
                    "fpts": as_int(settings.get("fpts")),
                    "fpts_against": as_int(settings.get("fpts_against")),
                    "fpts_against_decimal": as_int(settings.get("fpts_against_decimal")),
                    "fpts_decimal": as_int(settings.get("fpts_decimal")),
                    "losses": as_int(settings.get("losses")),
                    "ppts": as_int(settings.get("ppts")),
                    "ppts_decimal": as_int(settings.get("ppts_decimal")),
                    "ties": as_int(settings.get("ties")),
                    "total_moves": as_int(settings.get("total_moves")),
                    "waiver_budget_used": as_int(settings.get("waiver_budget_used")),
                    "waiver_position": as_int(settings.get("waiver_position")),
                    "wins": as_int(settings.get("wins")),
                },
            })
    if not rows:
        return empty_df(ROSTERS_SCHEMA)
    return spark.createDataFrame(rows, schema=ROSTERS_SCHEMA) \
                 .withColumn("_ingested_at", F.current_timestamp())

@dlt.table(
    name="sleeper_users_snapshot",
    comment="Users snapshot per league (point-in-time)"
)
def sleeper_users_snapshot():
    rows = []
    for lg in get_league_ids():
        url = f"https://api.sleeper.app/v1/league/{lg}/users"
        data = http_get_json(url) or []
        time.sleep(INTER_REQUEST_SLEEP_SECONDS)
        for u in data:
            rows.append({
                "league_id": as_str(lg),
                "user_id": as_str(u.get("user_id")),
                "display_name": as_str(u.get("display_name")),
                "metadata": as_str_map(u.get("metadata")),
                "is_owner": as_bool_or_none(u.get("is_owner")),
            })
    if not rows:
        return empty_df(USERS_SCHEMA)
    return spark.createDataFrame(rows, schema=USERS_SCHEMA) \
                 .withColumn("_ingested_at", F.current_timestamp())

@dlt.table(
    name="sleeper_transactions_snapshot",
    comment="Transactions snapshot per league × week (point-in-time)"
)
def sleeper_transactions_snapshot():
    rows = []
    for lg in get_league_ids():
        for wk in NFL_WEEKS:
            url = f"https://api.sleeper.app/v1/league/{lg}/transactions/{wk}"
            data = http_get_json(url) or []
            time.sleep(INTER_REQUEST_SLEEP_SECONDS)
            for t in data:
                rows.append({
                    "league_id": as_str(lg),
                    "transaction_id": as_str(t.get("transaction_id")),
                    "type": as_str(t.get("type")),
                    "status": as_str(t.get("status")),
                    "creator": as_str(t.get("creator")),
                    "adds": t.get("adds") or {},
                    "drops": t.get("drops") or {},
                    "roster_ids": t.get("roster_ids") or [],
                    "draft_picks": [json.dumps(x) for x in (t.get("draft_picks") or [])],
                    "waiver_bid": as_int(t.get("waiver_bid")),
                    "leg": as_int(t.get("leg")),
                    "metadata": as_str_map(t.get("metadata")),
                    "created": as_long(t.get("created")),
                })
    if not rows:
        return empty_df(TRANSACTIONS_SCHEMA)
    return spark.createDataFrame(rows, schema=TRANSACTIONS_SCHEMA) \
                 .withColumn("_ingested_at", F.current_timestamp())

@dlt.table(
    name="sleeper_matchups_snapshot",
    comment="Matchups snapshot per league × week (point-in-time)"
)
def sleeper_matchups_snapshot():
    rows = []
    for lg in get_league_ids():
        for wk in NFL_WEEKS:
            url = f"https://api.sleeper.app/v1/league/{lg}/matchups/{wk}"
            data = http_get_json(url) or []
            time.sleep(INTER_REQUEST_SLEEP_SECONDS)
            for m in data:
                rows.append({
                    "league_id": as_str(lg),
                    "week": as_int(wk),
                    "matchup_id": as_int(m.get("matchup_id")),
                    "roster_id": as_int(m.get("roster_id")),
                    "points": float(m.get("points")) if m.get("points") is not None else None,
                    "starters": m.get("starters") or [],
                    "players": m.get("players") or [],
                    "players_points": {str(k): float(v) for k, v in (m.get("players_points") or {}).items()} if m.get("players_points") else {},
                })
    if not rows:
        return empty_df(MATCHUPS_SCHEMA)
    return spark.createDataFrame(rows, schema=MATCHUPS_SCHEMA) \
                 .withColumn("_ingested_at", F.current_timestamp())

@dlt.table(
    name="sleeper_drafts_snapshot",
    comment="Drafts list per league (point-in-time)"
)
def sleeper_drafts_snapshot():
    rows = []
    for lg in get_league_ids():
        url = f"https://api.sleeper.app/v1/league/{lg}/drafts"
        data = http_get_json(url) or []
        time.sleep(INTER_REQUEST_SLEEP_SECONDS)
        for d in data:
            rows.append({
                "league_id": as_str(lg),
                "draft_id": as_str(d.get("draft_id")),
                "status": as_str(d.get("status")),
                "type": as_str(d.get("type")),
                "season": as_str(d.get("season")),
                "metadata": as_str_map(d.get("metadata")),
                "settings": as_str_map(d.get("settings")),
            })
    if not rows:
        return empty_df(DRAFTS_SCHEMA)
    return spark.createDataFrame(rows, schema=DRAFTS_SCHEMA) \
                 .withColumn("_ingested_at", F.current_timestamp())

@dlt.table(name="sleeper_draft_picks_snapshot",
           comment="Draft picks per draft (point-in-time)")
def sleeper_draft_picks_snapshot():
    rows = []
    for lg in get_league_ids():
        # fetch drafts HERE (don't rely on upstream collect)
        drafts = http_get_json(f"https://api.sleeper.app/v1/league/{lg}/drafts") or []
        time.sleep(INTER_REQUEST_SLEEP_SECONDS)
        for d in drafts:
            draft_id = as_str(d.get("draft_id"))
            if not draft_id:
                continue
            picks = http_get_json(f"https://api.sleeper.app/v1/draft/{draft_id}/picks") or []
            time.sleep(INTER_REQUEST_SLEEP_SECONDS)
            for p in picks:
                rows.append({
                    "draft_id": draft_id,
                    "league_id": as_str(lg),
                    "pick_no": as_int(p.get("pick_no")),
                    "round": as_int(p.get("round")),
                    "roster_id": as_int(p.get("roster_id")),
                    "player_id": as_str(p.get("player_id")),
                    "picked_by": as_str(p.get("picked_by")),
                    "is_keeper": as_bool_or_none(p.get("is_keeper")),
                    "metadata": as_str_map(p.get("metadata")),
                })
    if not rows:
        return empty_df(DRAFT_PICKS_SCHEMA)
    return spark.createDataFrame(rows, schema=DRAFT_PICKS_SCHEMA) \
                 .withColumn("_ingested_at", F.current_timestamp())

@dlt.table(name="sleeper_draft_slot_to_roster_snapshot",
           comment="Draft slot to roster_id mapping per draft (point-in-time)")
def sleeper_draft_slot_to_roster_snapshot():
    rows = []
    for lg in get_league_ids():
        # fetch drafts HERE (don't rely on upstream collect)
        drafts = http_get_json(f"https://api.sleeper.app/v1/league/{lg}/drafts") or []
        time.sleep(INTER_REQUEST_SLEEP_SECONDS)
        for d in drafts:
            draft_id = as_str(d.get("draft_id"))
            if not draft_id:
                continue
            # Fetch the full draft details to get slot_to_roster_id
            draft_details = http_get_json(f"https://api.sleeper.app/v1/draft/{draft_id}") or {}
            time.sleep(INTER_REQUEST_SLEEP_SECONDS)
            
            # slot_to_roster_id is a map of slot -> roster_id
            slot_to_roster_id = draft_details.get("slot_to_roster_id") or {}
            for slot_str, roster_id in slot_to_roster_id.items():
                rows.append({
                    "draft_id": draft_id,
                    "league_id": as_str(lg),
                    "slot": as_int(slot_str),
                    "roster_id": as_int(roster_id),
                })
    if not rows:
        return empty_df(DRAFT_SLOT_TO_ROSTER_SCHEMA)
    return spark.createDataFrame(rows, schema=DRAFT_SLOT_TO_ROSTER_SCHEMA) \
                 .withColumn("_ingested_at", F.current_timestamp())

